# Assignment 9: Transformer NMT with FastText and BPEmb embeddings (OpenNMT-py)
Same dataset, length bins and train/valid/test splits as Assignment 8. Two Transformer models are trained, one initialised with **FastText word embeddings** and one with **BPEmb subword embeddings**, and both are evaluated with bin-wise sacreBLEU.

In [1]:
# !pip install -q OpenNMT-py sacrebleu bpemb sentencepiece gensim requests pandas pyyaml

## 1. Configuration and data (splits from Assignment 8)

In [2]:
import os, sys, re, glob, subprocess
import pandas as pd

SPLIT_DIR = "work"       # train/valid/test CSVs written by the Assignment 8 notebook (same dataset, same bins)
WORK = "work9"
SEED = 42
EMB_DIM = 300            # FastText wiki vectors are 300-d, so BPEmb is also used at 300-d
VOCAB_WORD = 30000       # word vocabulary size for the FastText model
BPE_VS = 10000           # BPEmb subword vocabulary size
TRAIN_STEPS = 10000
os.makedirs(WORK, exist_ok=True)

for n in ["train", "valid", "test"]:
    assert os.path.exists(f"{SPLIT_DIR}/{n}.csv"), f"{SPLIT_DIR}/{n}.csv not found - run the Assignment 8 notebook first"
train, valid, test = [pd.read_csv(f"{SPLIT_DIR}/{n}.csv", keep_default_na=False) for n in ["train", "valid", "test"]]
print(pd.DataFrame({"train": train["bin"].value_counts(), "valid": valid["bin"].value_counts(),
                    "test": test["bin"].value_counts()}).sort_index())

      train  valid  test
bin                     
bin1  29370    667  3337
bin2  66515   1511  7558
bin3  32289    733  3669
bin4  17600    400  2000


## 2. Shared helpers (OpenNMT config, training, translation, bin-wise BLEU)

In [3]:
import torch, yaml, sacrebleu
use_gpu = torch.cuda.is_available()
print("GPU available:", use_gpu)

def write_lines(lines, path):
    with open(path, "w", encoding="utf-8", newline="\n") as f:      # LF endings (Windows CRLF breaks OpenNMT vocab files)
        f.write("\n".join(lines) + "\n")

def to_lf(path):
    with open(path, "rb") as f: b = f.read()
    if b"\r" in b:
        with open(path, "wb") as f: f.write(b.replace(b"\r\n", b"\n"))

def run_cmd(args):
    """Run `python -m <args>` and stream its output into the cell."""
    p = subprocess.Popen([sys.executable, "-m"] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace",
                         env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
                              "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:128,garbage_collection_threshold:0.8"})
    for line in p.stdout:
        if "Weighted corpora" in line or "* corpus_1" in line: continue
        print(line, end="")
    p.wait()
    if p.returncode != 0: raise RuntimeError(f"command failed (exit code {p.returncode})")

def make_cfg(tag, vocab_size, path, src_emb=None, tgt_emb=None):
    d = f"{WORK}/{tag}"
    cfg = {
        "save_data": f"{d}/onmt", "src_vocab": f"{d}/vocab.src", "tgt_vocab": f"{d}/vocab.tgt", "overwrite": True,
        "src_vocab_size": vocab_size, "tgt_vocab_size": vocab_size,
        "data": {"corpus_1": {"path_src": f"{d}/train.src", "path_tgt": f"{d}/train.tgt"},
                 "valid":    {"path_src": f"{d}/valid.src", "path_tgt": f"{d}/valid.tgt"}},
        "save_model": f"{d}/model", "save_checkpoint_steps": 2500, "keep_checkpoint": 3,
        "train_steps": TRAIN_STEPS, "valid_steps": 2500, "report_every": 200, "early_stopping": 0, "seed": SEED,
        "world_size": 1 if use_gpu else 0, "gpu_ranks": [0] if use_gpu else [],
        "encoder_type": "transformer", "decoder_type": "transformer", "position_encoding": True,
        "enc_layers": 4, "dec_layers": 4, "heads": 6, "self_attn_type": "scaled-dot",   # head size 50 is not supported by the fused attention kernel
        "hidden_size": EMB_DIM, "word_vec_size": EMB_DIM, "transformer_ff": 4 * EMB_DIM,
        "dropout": [0.1], "attention_dropout": [0.1], "label_smoothing": 0.1,
        "param_init": 0.0, "param_init_glorot": True,
        "optim": "adam", "adam_beta1": 0.9, "adam_beta2": 0.998,
        "decay_method": "noam", "warmup_steps": 4000, "learning_rate": 2.0, "max_grad_norm": 0.0,
        "batch_size": 1024, "batch_type": "tokens", "normalization": "tokens", "accum_count": [8],   # small batches for a 4 GB GPU
        "max_generator_batches": 2,
    }
    if src_emb:   # pretrained embeddings (word2vec text format) for encoder and decoder inputs
        cfg.update({"src_embeddings": src_emb, "tgt_embeddings": tgt_emb, "embeddings_type": "word2vec"})
    with open(path, "w") as f: yaml.safe_dump(cfg, f)

def build_vocab(tag):
    d = f"{WORK}/{tag}"
    if not (os.path.exists(f"{d}/vocab.src") and os.path.exists(f"{d}/vocab.tgt")):
        run_cmd(["onmt.bin.build_vocab", "-config", f"{d}/vocab.yaml", "-n_sample", "-1"])
    for p in [f"{d}/vocab.src", f"{d}/vocab.tgt"]: to_lf(p)

def read_vocab(path, n):
    with open(path, encoding="utf-8", newline="") as f:
        toks = [l.rstrip("\n").split("\t")[0] for l in f]
    return [t for t in toks if t][:n]

def train_model(tag):
    d = f"{WORK}/{tag}"
    final = f"{d}/model_step_{TRAIN_STEPS}.pt"
    ckpts = sorted(glob.glob(f"{d}/model_step_*.pt"), key=lambda p: int(re.findall(r"step_(\d+)", p)[0]))
    if os.path.exists(final):
        print("Training already finished -> skipping"); return
    args = ["onmt.bin.train", "-config", f"{d}/train.yaml"]
    if ckpts:
        print("Resuming from", ckpts[-1]); args += ["-train_from", ckpts[-1]]
    run_cmd(args)

def translate_test(tag):
    d = f"{WORK}/{tag}"
    if not os.path.exists(f"{d}/test.pred"):
        run_cmd(["onmt.bin.translate", "-model", f"{d}/model_step_{TRAIN_STEPS}.pt", "-src", f"{d}/test.src",
                 "-output", f"{d}/test.pred", "-beam_size", "5", "-batch_size", "64", "-max_length", "256"]
                + (["-gpu", "0"] if use_gpu else []))
    with open(f"{d}/test.pred", encoding="utf-8") as f:
        pred = [l.rstrip("\n") for l in f]
    assert len(pred) == len(test), (len(pred), len(test))
    return pred

def bin_bleu(hyps, name):
    df = test.copy(); df["hyp"] = hyps
    rows = []
    for b in sorted(df["bin"].unique()):
        g = df[df["bin"] == b]
        rows.append({"bin": b, "#test sents": len(g), name: round(sacrebleu.corpus_bleu(g["hyp"].tolist(), [g["tgt"].tolist()]).score, 2)})
    overall = sacrebleu.corpus_bleu(df["hyp"].tolist(), [df["tgt"].tolist()])
    rows.append({"bin": "overall", "#test sents": len(df), name: round(overall.score, 2)})
    print(name, "->", overall)
    return pd.DataFrame(rows)

GPU available: True


## 3. Model 1: FastText word embeddings
Text is split into words (punctuation separated). English and Hindi FastText vectors (`wiki.en.vec`, `wiki.hi.vec`, 300-d) are streamed from fasttext.cc and only the vectors for words in our vocabulary are kept. These vectors initialise the encoder and decoder embeddings.

In [4]:
import requests
FT = f"{WORK}/fasttext"; os.makedirs(FT, exist_ok=True)

PUNCT = re.compile(r'([.,!?;:()\[\]{}"“”‘’।|…])')
def word_tok(s): return PUNCT.sub(r" \1 ", s).split()

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    write_lines([" ".join(word_tok(s)) for s in d["src"]], f"{FT}/{name}.src")
    if name != "test":
        write_lines([" ".join(word_tok(s)) for s in d["tgt"]], f"{FT}/{name}.tgt")

make_cfg("fasttext", VOCAB_WORD, f"{FT}/vocab.yaml")
build_vocab("fasttext")
print(read_vocab(f"{FT}/vocab.src", 8)); print(read_vocab(f"{FT}/vocab.tgt", 8))

['.', 'the', ',', 'of', 'and', 'is', 'in', 'to']
['।', 'है', 'के', ',', 'में', 'की', 'से', 'और']


In [5]:
FT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{}.vec"

def extract_fasttext(lang, vocab_tokens, out_path):
    """Stream wiki.<lang>.vec and keep only vectors for our vocabulary (falls back to the lower-cased word)."""
    if os.path.exists(out_path):
        print(out_path, "already exists -> skipping"); return
    wanted = set(vocab_tokens) | {t.lower() for t in vocab_tokens}
    found = {}
    with requests.get(FT_URL.format(lang), stream=True, timeout=120) as r:
        r.raise_for_status()
        for i, raw in enumerate(r.iter_lines(chunk_size=1 << 20, delimiter=b"\n")):
            if i == 0: continue                                  # header line
            sp = raw.find(b" ")
            if sp < 0: continue
            w = raw[:sp].decode("utf-8", "ignore")
            if w in wanted and w not in found:
                vec = raw[sp + 1:].decode("utf-8", "ignore").split()
                if len(vec) == EMB_DIM: found[w] = vec
            if i % 500000 == 0: print(f"  {lang}: read {i:,} vectors, kept {len(found):,}")
    rows = [(t, found.get(t) or found.get(t.lower())) for t in vocab_tokens]
    rows = [(t, v) for t, v in rows if v]
    with open(out_path, "w", encoding="utf-8", newline="\n") as f:
        f.write(f"{len(rows)} {EMB_DIM}\n")
        for t, v in rows: f.write(t + " " + " ".join(v) + "\n")
    print(f"{lang}: {len(rows):,} of {len(vocab_tokens):,} vocabulary words have a FastText vector")

src_vocab_ft = read_vocab(f"{FT}/vocab.src", VOCAB_WORD)
tgt_vocab_ft = read_vocab(f"{FT}/vocab.tgt", VOCAB_WORD)
extract_fasttext("en", src_vocab_ft, f"{FT}/emb.src.vec")
extract_fasttext("hi", tgt_vocab_ft, f"{FT}/emb.tgt.vec")

work9/fasttext/emb.src.vec already exists -> skipping
work9/fasttext/emb.tgt.vec already exists -> skipping


In [6]:
make_cfg("fasttext", VOCAB_WORD, f"{FT}/train.yaml", src_emb=f"{FT}/emb.src.vec", tgt_emb=f"{FT}/emb.tgt.vec")
train_model("fasttext")

Training already finished -> skipping


In [7]:
hyp_ft = translate_test("fasttext")
res_ft = bin_bleu(hyp_ft, "FastText BLEU")
res_ft

FastText BLEU -> BLEU = 23.77 51.8/29.2/18.0/11.7 (BP = 1.000 ratio = 1.029 hyp_len = 365955 ref_len = 355590)


,bin,#test sents,FastText BLEU
0,bin1,3337,27.39
1,bin2,7558,23.98
2,bin3,3669,22.29
3,bin4,2000,24.04
4,overall,16564,23.77


## 4. Model 2: BPEmb subword embeddings
Text is first encoded into subwords with BPEmb's own SentencePiece BPE models (English and Hindi), as described in the BPEmb repository. The matching pretrained BPEmb vectors initialise the encoder and decoder embeddings. Output subwords are decoded back to text with BPEmb.
BPEmb (SentencePiece) runs in a separate Python process so that a native crash cannot kill the notebook kernel.

In [8]:
BP = f"{WORK}/bpemb"; os.makedirs(BP, exist_ok=True)

BPEMB_TOOL = f"{BP}/bpemb_tool.py"
with open(BPEMB_TOOL, "w", encoding="utf-8") as f:
    f.write(r"""
import sys
from bpemb import BPEmb
cmd, lang, vs, dim = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
m = BPEmb(lang=lang, vs=vs, dim=dim)
if cmd == "vectors":      # BPEmb vectors in word2vec text format (loadable by OpenNMT)
    with open(sys.argv[5], "w", encoding="utf-8", newline="\n") as f:
        f.write(f"{len(m.pieces)} {m.dim}\n")
        for p, v in zip(m.pieces, m.vectors):
            f.write(p + " " + " ".join(f"{x:.6f}" for x in v) + "\n")
else:
    with open(sys.argv[5], encoding="utf-8") as fin, open(sys.argv[6], "w", encoding="utf-8", newline="\n") as fout:
        lines = [l.rstrip("\n") for l in fin]
        if cmd == "encode":
            for i in range(0, len(lines), 10000):
                for pieces in m.encode(lines[i:i + 10000]):
                    fout.write(" ".join(pieces) + "\n")
        else:             # decode
            for l in lines:
                fout.write((m.decode(l.split()) if l.split() else "") + "\n")
""")

def run_bpemb(cmd, lang, *paths):
    p = subprocess.Popen([sys.executable, BPEMB_TOOL, cmd, lang, str(BPE_VS), str(EMB_DIM), *paths],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
                         env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"})
    out = p.communicate()[0]
    if p.returncode != 0:
        print(out[-3000:]); raise RuntimeError(f"bpemb step failed (exit code {p.returncode})")

clean = lambda col: [" ".join(str(s).split()) for s in col]
for name, d in [("train", train), ("valid", valid), ("test", test)]:
    write_lines(clean(d["src"]), f"{BP}/{name}.raw.src")
    run_bpemb("encode", "en", f"{BP}/{name}.raw.src", f"{BP}/{name}.src")
    if name != "test":
        write_lines(clean(d["tgt"]), f"{BP}/{name}.raw.tgt")
        run_bpemb("encode", "hi", f"{BP}/{name}.raw.tgt", f"{BP}/{name}.tgt")
print(open(f"{BP}/train.src", encoding="utf-8").readline())
print(open(f"{BP}/train.tgt", encoding="utf-8").readline())

▁you ▁are ▁doing ▁some ▁def ault ▁reason ing ▁on ▁the ▁way .

▁आप ▁रास्ते ▁में ▁कुछ ▁डि फ़ ॉल ्ट ▁तर्क ▁कर ▁रहे ▁हैं ।



In [9]:
run_bpemb("vectors", "en", f"{BP}/emb.src.vec")
run_bpemb("vectors", "hi", f"{BP}/emb.tgt.vec")

make_cfg("bpemb", BPE_VS, f"{BP}/vocab.yaml")
build_vocab("bpemb")
make_cfg("bpemb", BPE_VS, f"{BP}/train.yaml", src_emb=f"{BP}/emb.src.vec", tgt_emb=f"{BP}/emb.tgt.vec")
train_model("bpemb")

Corpus corpus_1's weight should be given. We default it to 1 for you.
[2026-10-27 18:20:22,358 INFO] Counter vocab from -1 samples.
[2026-10-27 18:20:22,359 INFO] n_sample=-1: Build vocab on full datasets.
[2026-10-27 18:20:34,849 INFO] Counters src: 9757
[2026-10-27 18:20:34,849 INFO] Counters tgt: 9726
[2026-10-27 18:20:40,223 INFO] Missing transforms field for corpus_1 data, set to default: [].
[2026-10-27 18:20:40,224 WARNING] Corpus corpus_1's weight should be given. We default it to 1 for you.
[2026-10-27 18:20:40,224 INFO] Missing transforms field for valid data, set to default: [].
[2026-10-27 18:20:40,224 INFO] Parsed 2 corpora from -data.
[2026-10-27 18:20:40,224 INFO] Get special vocabs from Transforms: {'src': [], 'tgt': []}.
[2026-10-27 18:20:40,262 INFO] Reading encoder embeddings from work9/bpemb/emb.src.vec
[2026-10-27 18:20:40,929 INFO] 	Found 10000 total vectors in file.
[2026-10-27 18:20:40,929 INFO] Reading decoder embeddings from work9/bpemb/emb.tgt.vec
[2026-10-27

In [10]:
pred_bp = translate_test("bpemb")
write_lines(pred_bp, f"{BP}/test.pred.sp")
run_bpemb("decode", "hi", f"{BP}/test.pred.sp", f"{BP}/test.pred.txt")
with open(f"{BP}/test.pred.txt", encoding="utf-8") as f:
    hyp_bp = [l.rstrip("\n") for l in f]
res_bp = bin_bleu(hyp_bp, "BPEmb BLEU")
res_bp

[2026-10-27 19:43:51,147 INFO] Loading checkpoint from work9/bpemb/model_step_10000.pt
[2026-10-27 19:43:51,961 INFO] Loading data into the model
c:\Users\dhruv\Downloads\ILLM_SVNIT\venv\Lib\site-packages\onmt\modules\multi_headed_attn.py:656: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(
[2026-10-27 19:45:19,208 INFO] PRED SCORE: -0.4817, PRED PPL: 1.62 NB SENTENCES: 16564
Time w/o python interpreter load/terminate:  88.07191228866577
BPEmb BLEU -> BLEU = 23.51 56.0/31.1/18.7/11.9 (BP = 0.943 ratio = 0.945 hyp_len = 335906 ref_len = 355590)


,bin,#test sents,BPEmb BLEU
0,bin1,3337,27.53
1,bin2,7558,23.78
2,bin3,3669,22.76
3,bin4,2000,22.78
4,overall,16564,23.51


## 5. Bin-wise BLEU comparison

In [11]:
results = res_ft.merge(res_bp[["bin", "BPEmb BLEU"]], on="bin")
results

,bin,#test sents,FastText BLEU,BPEmb BLEU
0,bin1,3337,27.39,27.53
1,bin2,7558,23.98,23.78
2,bin3,3669,22.29,22.76
3,bin4,2000,24.04,22.78
4,overall,16564,23.77,23.51


## 6. Bin-wise result analysis
*(Fill in after running; refer to the table above.)*

- **Trend across bins** for each embedding type, and which bin is best / worst.
- **FastText vs BPEmb:** which is better overall and per bin. Word-level vocabularies leave rare and unseen words as `<unk>`, while subwords can represent any word.
- **Notes:** BPEmb lower-cases the input and maps digits to 0 (its standard preprocessing), and both systems are scored against the same raw references.
- **Caveats:** single run per model, one reference per sentence.